# ME 313 · Lab 4.1 — Read *h* off a cooling curve
**Module 4 companion (Transient Conduction).**

A small lumped body cooling in a stream follows $\theta/\theta_i = e^{-t/\tau}$ with $\tau=\rho V c/(hA_s)=\rho L_c c/h$. 
That means a **measured cooling curve secretly contains $h$** — an *inverse problem* you can solve with a curve fit. 
You generate the data (as if from a sensor), let a fitter recover $\tau$, then back out $h$ and **check it against the known value**.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import curve_fit

### 1. A cooling experiment (synthetic sensor data)
An aluminium spherical sensor ($D=3$ mm) cooling from 80 °C in 20 °C air, sampled every ~2 s with noise.


In [ ]:
rng = np.random.default_rng(0)
rho, c, k = 2700., 900., 237.       # aluminium (SI)
D = 3e-3; r0 = D/2; Lc = r0/3        # sphere: Lc = V/As = r0/3
h_true = 60.0                        # the value we will pretend not to know
tau_true = rho*Lc*c/h_true
Ti, Tinf = 80.0, 20.0
t = np.linspace(0, 90, 40)
theta = (Ti-Tinf)*np.exp(-t/tau_true) + Tinf
T_meas = theta + rng.normal(0, 0.8, t.shape)   # sensor noise
print(f'true tau = {tau_true:.2f} s   (h_true = {h_true} W/m^2K)')

### 2. Fit the exponential to recover $\tau$
We fit $T(t)=(T_i-T_\infty)e^{-t/\tau}+T_\infty$, treating **both** $\tau$ and $T_\infty$ as unknowns.


In [ ]:
def model(t, tau, Tinf_):
    return (Ti - Tinf_)*np.exp(-t/tau) + Tinf_

(tau_fit, Tinf_fit), _ = curve_fit(model, t, T_meas, p0=[10, 25])
print(f'fitted tau  = {tau_fit:.2f} s   (true {tau_true:.2f})')
print(f'fitted Tinf = {Tinf_fit:.1f} C')

### 3. Back out $h$ and check the physics
From $\tau=\rho L_c c/h$ we get $h=\rho L_c c/\tau$.


In [ ]:
h_fit = rho*Lc*c/tau_fit
err = 100*abs(h_fit-h_true)/h_true
print(f'recovered h = {h_fit:.1f} W/m^2K   (true {h_true})   error {err:.1f}%')

plt.figure(figsize=(6,4))
plt.plot(t, T_meas, 'o', ms=4, label='measured')
plt.plot(t, model(t, tau_fit, Tinf_fit), '-', label='fit')
plt.xlabel('t (s)'); plt.ylabel('T (C)'); plt.legend(); plt.title('Cooling-curve fit'); plt.grid(alpha=.3)
plt.show()

### 4. Verify against the physics (the point of the lab)
The recovered $h$ should match the value used to generate the data to within the noise. 
You just solved an **inverse heat-transfer problem** — the same idea used to characterise materials (recover $\alpha$) or calibrate sensors from a transient.


### Your turn
1. Increase the noise (0.8 → 3 °C). How much does the recovered $h$ drift? How many samples restore accuracy?
2. Change the body to a **copper** sphere or a different diameter and confirm you still recover the input $h$.
3. **AI companion:** ask an LLM to derive $\tau$ for this body, then check its algebra against your fit.
4. Rework the fit to recover the **thermal diffusivity $\alpha$** instead of $h$, given a known $h$.
